### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [ ]:
# rows: sub_id|client_name|plan|visits_left|status
rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active',
]


class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        # TODO: сохранить sub_id, client_name, plan
        # TODO: visits_left хранить через self._visits_left
        # TODO: значение visits_left пропустить через property/setter
        # TODO: проверить plan и status, иначе raise ValueError
        self.sub_id = sub_id
        self.client_name = client_name
        if plan in self.allowed_plans:
            self.plan = plan
        else:
            raise ValueError('некорректный план!')
        if status in self.allowed_statuses:
            self.status = status
        else:
            raise ValueError('неккоректный статус!')
        self._visits_left = visits_left
        

    @property
    def visits_left(self):
        # TODO: вернуть текущее число посещений
        return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        # TODO: привести value к int
        # TODO: если value < 0 -> raise ValueError('Visits must be >= 0')
        # TODO: сохранить результат в self._visits_left
        value = int(value)
        if value < 0:
            raise ValueError('Visits must be >= 0')
        self._visits_left = value
    def use_visit(self):
        # TODO: если статус не 'active' -> raise ValueError
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: уменьшить visits_left на 1
        # TODO: если после списания visits_left == 0, перевести статус в 'expired'
        if self.status != 'active':
            raise ValueError('статус должен быть active!')
        if self._visits_left == 0:
            raise ValueError('нулевой остаток!')
        self._visits_left -= 1
        if self._visits_left == 0:
            self.status = 'expired'

    def freeze(self):
        # TODO: если статус 'expired' -> raise ValueError
        # TODO: перевести абонемент в 'frozen'
        if self.status == 'expired':
            raise ValueError('статус не должен быть expired!')
        self.status = 'frozen'

    def activate(self):
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: перевести абонемент в 'active'
        if self._visits_left == 0:
            raise ValueError('нулевой остаток!')
        self.status = 'active'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: sub_id, client_name, plan, visits_left, status
        # TODO: вернуть Subscription(...)
        arr = row.split('|')
        sub_id, client_name, plan, visits_left, status = arr[0], arr[1], arr[2], int(arr[3]), arr[4]
        return Subscription(status=status, sub_id=sub_id, visits_left=visits_left, plan=plan, client_name=client_name)

    def __repr__(self):
        # TODO: вернуть строку вида Subscription(sub_id='...', client_name='...', status='...')
        return f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', status='{self.status}')"


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        # TODO: добавить subscription в self.items
        self.items.append(subscription)

    def load(self, rows):
        # TODO: для каждой строки создать Subscription.from_row(row)
        # TODO: добавить объект в реестр через add(...)
        for el in rows:
            res = Subscription.from_row(el)
            self.add(res)

    def active_subscriptions(self):
        # TODO: вернуть список абонементов со статусом 'active'
        res = []
        for el in self.items:
            if el.status == 'active':
                res.append(el)
        return res
    def by_plan(self, plan):
        # TODO: вернуть список абонементов нужного тарифа
        res = []
        for el in self.items:
            if el.plan == plan:
                res.append(el)
        return res

    def total_visits_left(self):
        # TODO: вернуть суммарное число оставшихся посещений
        m = 0
        for el in self.items:
            m += el._visits_left
        return m

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        obj = {}
        for el in self.items:
            obj.setdefault(el.status, 0)
            obj[el.status] += el._visits_left
        return obj

    def find(self, sub_id):
        # TODO: вернуть абонемент по sub_id или None
        for el in self.items:
            if el.sub_id == sub_id:
                return el
        return None



registry = SubscriptionRegistry()

# TODO: загрузить rows в registry
# TODO: вывести все абонементы
# TODO: вывести active_subscriptions()
# TODO: вывести by_plan('standard')
# TODO: вывести total_visits_left()
# TODO: вывести status_summary()
# TODO: найти абонемент 'SB-101', активировать его и вывести status_summary()
# TODO: найти абонемент 'SB-100', списать одно посещение и вывести объект
registry.load(rows)
print(f'Все абонементы: {registry.items}')
print('----------------------------')
print(registry.active_subscriptions())
print('----------------------------')
print(registry.by_plan('standard'))
print('----------------------------')
print(registry.total_visits_left())
print('----------------------------')
print(registry.status_summary())
ab = registry.find('SB-101')
ab.activate()
print(registry.status_summary())
ab = registry.find('SB-100')
ab.use_visit()
print(ab)
